# 03-2. 출시 전 LLM 분석 결과 품질 점검

## 점검 목적
| 구분 | 확인 내용 |
|---|---|
| 결과 파일 확인 | LLM 분석 결과 파일과 이슈 태그 파일이 정상적으로 불러와지는지 확인 |
| 기본 품질 점검 | 분석 결과 행 수, 분석 게임 수, 분석 성공 여부, 중복 리뷰 ID, 결측치 확인 |
| 필터 조건 확인 | D0-D30 출시 초기 리뷰만 포함되었는지, Steam 긍정/부정 라벨 분포가 어떤지 확인 |
| LLM 감정 결과 확인 | LLM 감정 분포와 Steam 라벨 간의 관계를 확인 |
| 이슈 분류 확인 | 주요 이슈와 시급도 분포를 확인하여 반복적으로 나타나는 문제 유형 파악 |
| 이슈 태그 확인 | 리뷰별 이슈 태그가 정상적으로 추출되었는지, 태그 누락이 있는지 확인 |
| 게임별 샘플 확인 | 게임별 리뷰 수, 긍정/부정 리뷰 수, 평균 감정 점수 확인 |
| 최종 요약 | 분석 결과가 보고서에 활용 가능한 수준인지 핵심 지표로 정리 |

# 0. 기본 라이브러리

In [1]:
import pandas as pd
from pathlib import Path

# 1. 파일 경로 설정

`03-1_run_llm_sentiment_Action_PreLaunch_v1.ipynb`와 같은 저장 위치를 사용한다.  
다른 환경에서 실행할 경우 `ROOT`만 본인 프로젝트 경로에 맞게 수정하면 된다.

In [2]:
# 프로젝트 루트 직접 지정
# 다른 환경에서 실행할 경우 ROOT만 본인 프로젝트 경로에 맞게 수정합니다.
ROOT = Path(r"C:\Users\joon5\Documents\github\steam-indie-game-analysis")

# ============================================================
# LLM 분석 산출물 폴더
# ============================================================
RUN_NAME = "main_action_d0-d30_1000_v2"
OUTPUT_DIR = ROOT / "data" / "outputs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# LLM 분석 산출물
RESULT_CSV_PATH = OUTPUT_DIR / "llm_review_analysis_result.csv"
ISSUE_TAG_FLAT_PATH = OUTPUT_DIR / "llm_issue_tags_flat.csv"

# 선택 산출물
GAME_SUMMARY_PATH = OUTPUT_DIR / "llm_game_summary.csv"
ISSUE_PRIORITY_PATH = OUTPUT_DIR / "llm_issue_priority_summary.csv"

print("\n프로젝트 루트:", ROOT)
print("전처리 후보 데이터:", RESULT_CSV_PATH)
print("전처리 게임 요약:", ISSUE_TAG_FLAT_PATH)
print("결과 저장 폴더:", OUTPUT_DIR)


프로젝트 루트: C:\Users\joon5\Documents\github\steam-indie-game-analysis
전처리 후보 데이터: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0-d30_1000_v2\llm_review_analysis_result.csv
전처리 게임 요약: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0-d30_1000_v2\llm_issue_tags_flat.csv
결과 저장 폴더: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\outputs\main_action_d0-d30_1000_v2


# 2. 결과 파일 불러오기

LLM 분석 결과 파일과 이슈 태그 펼친 파일을 불러온다.

In [3]:
# ============================================================
# 2. 데이터 불러오기
# ============================================================
df_result = pd.read_csv(RESULT_CSV_PATH)
df_tag = pd.read_csv(ISSUE_TAG_FLAT_PATH)

print("LLM 분석 결과 행 수:", len(df_result))
print("이슈 태그 행 수:", len(df_tag))
print("분석 게임 수:", df_result["appid"].nunique())

LLM 분석 결과 행 수: 770
이슈 태그 행 수: 1552
분석 게임 수: 28


# 3. 기본 품질 점검

분석 성공 여부, 중복 리뷰 ID, 컬럼별 결측치를 확인한다.

In [4]:
# ============================================================
# 3. 기본 품질 점검
# ============================================================

# LLM 분석 상태 확인
print("[분석 상태]")
display(df_result["analysis_status"].value_counts(dropna=False))

# 리뷰 ID 중복 여부 확인
print("[중복 리뷰 ID 개수]")
print(df_result["recommendationid"].duplicated().sum())

# 컬럼별 결측치 확인
print("[컬럼별 결측치]")
missing = df_result.isna().sum()
display(missing[missing > 0].sort_values(ascending=False))

[분석 상태]


analysis_status
success    770
Name: count, dtype: int64

[중복 리뷰 ID 개수]
0
[컬럼별 결측치]


Series([], dtype: int64)

# 4. 필터 조건 확인

출시 초기 구간, 출시 후 경과일, Steam 긍정/부정 라벨 분포를 확인한다.

In [5]:
# ============================================================
# 4. 필터 조건 확인
# ============================================================

# 리뷰가 의도한 출시 초기 구간에 포함되어 있는지 확인
print("[리뷰 기간 분포]")
display(df_result["release_period"].value_counts(dropna=False))

# 출시일 기준 경과일 범위 확인
print("[출시 후 경과일 범위]")
print("최소:", df_result["days_from_release"].min())
print("최대:", df_result["days_from_release"].max())

# Steam 원본 라벨 기준 긍정/부정 개수 확인
print("[Steam 긍정/부정 라벨 분포]")
display(df_result["steam_label_text"].value_counts(dropna=False))

# Steam 원본 라벨 기준 긍정/부정 비율 확인
print("[Steam 긍정/부정 라벨 비율]")
display(
    (df_result["steam_label_text"].value_counts(normalize=True) * 100)
    .round(1)
    .rename("ratio_percent")
)

[리뷰 기간 분포]


release_period
D0-D30    770
Name: count, dtype: int64

[출시 후 경과일 범위]
최소: 0.0
최대: 30.0
[Steam 긍정/부정 라벨 분포]


steam_label_text
positive    511
negative    259
Name: count, dtype: int64

[Steam 긍정/부정 라벨 비율]


steam_label_text
positive    66.4
negative    33.6
Name: ratio_percent, dtype: float64

# 5. LLM 감정 분석 결과 확인

LLM이 판단한 감정 분포와 Steam 라벨 간의 관계를 확인한다.

In [6]:
# ============================================================
# 5. LLM 감정 분석 결과 확인
# ============================================================

# LLM 감정 분포 확인
print("[LLM 감정 분포]")
display(df_result["llm_sentiment"].value_counts(dropna=False))

# Steam 라벨과 LLM 감정 판단의 관계 확인
print("[Steam 라벨과 LLM 감정 관계]")
display(df_result["sentiment_relation"].value_counts(dropna=False))

# Steam 라벨과 LLM 감정 판단의 교차표 확인
print("[Steam 라벨 × LLM 감정 교차표]")
display(
    pd.crosstab(
        df_result["steam_label_text"],
        df_result["llm_sentiment"],
        margins=True
    )
)

[LLM 감정 분포]


llm_sentiment
positive    465
negative    241
mixed        58
neutral       6
Name: count, dtype: int64

[Steam 라벨과 LLM 감정 관계]


sentiment_relation
exact_match      704
partial_match     58
unclear            6
mismatch           2
Name: count, dtype: int64

[Steam 라벨 × LLM 감정 교차표]


llm_sentiment,mixed,negative,neutral,positive,All
steam_label_text,,,,,
negative,19,239,1,0,259
positive,39,2,5,465,511
All,58,241,6,465,770


# 6. 주요 이슈 분포 확인

LLM이 분류한 주요 이슈와 시급도 분포를 확인한다.

In [7]:
# ============================================================
# 6. 주요 이슈 분포 확인
# ============================================================

# 리뷰별 대표 이슈 분포 확인
print("[Primary Issue 분포]")
display(df_result["primary_issue"].value_counts(dropna=False))

# 개선 시급도 분포 확인
print("[Urgency 분포]")
display(df_result["urgency"].value_counts(dropna=False))

[Primary Issue 분포]


primary_issue
positive_praise            345
gameplay_loop              124
bug                         46
content_volume              33
difficulty                  31
control                     30
balance                     23
other                       18
ui_ux                       17
crash                       16
performance                 14
progression_grind           14
graphics_audio              13
save_progression            12
optimization                10
story                        8
multiplayer_network          6
developer_communication      6
price_value                  4
Name: count, dtype: int64

[Urgency 분포]


urgency
low       402
high      214
medium    154
Name: count, dtype: int64

# 7. 이슈 태그 품질 확인

리뷰별 이슈 태그 개수, 태그가 없는 리뷰 수, 태그 카테고리 분포를 확인한다.

In [8]:
# ============================================================
# 7. 이슈 태그 품질 확인
# ============================================================

# 리뷰별로 몇 개의 이슈 태그가 추출되었는지 확인
tag_count_by_review = (
    df_tag
    .groupby("recommendationid")
    .size()
    .reset_index(name="tag_count")
)

print("[리뷰별 이슈 태그 개수 요약]")
display(tag_count_by_review["tag_count"].describe())

# LLM 분석 결과에는 있지만 이슈 태그 파일에는 없는 리뷰 확인
reviews_without_tags = set(df_result["recommendationid"]) - set(df_tag["recommendationid"])

print("[이슈 태그가 없는 리뷰 수]")
print(len(reviews_without_tags))

if len(reviews_without_tags) > 0:
    display(df_result[df_result["recommendationid"].isin(reviews_without_tags)])

# 태그 카테고리 분포 확인
print("[태그 카테고리 분포]")
display(df_tag["tag_category"].value_counts(dropna=False))

[리뷰별 이슈 태그 개수 요약]


count    770.000000
mean       2.015584
std        1.001827
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max        6.000000
Name: tag_count, dtype: float64

[이슈 태그가 없는 리뷰 수]
0
[태그 카테고리 분포]


tag_category
positive_praise             463
gameplay_loop               247
difficulty                   98
bug                          95
graphics_audio               82
control                      76
content_volume               70
balance                      67
ui_ux                        64
other                        51
story                        50
progression_grind            36
price_value                  25
performance                  24
developer_communication      23
save_progression             23
optimization                 21
crash                        20
multiplayer_network          14
translation_localization      3
Name: count, dtype: int64

# 8. 게임별 샘플 수 확인

게임별로 몇 개의 리뷰가 LLM 분석에 사용되었는지 확인한다.

In [9]:
# ============================================================
# 8. 게임별 샘플 수 확인
# ============================================================

game_sample_summary = (
    df_result
    .groupby(["appid", "game_name"])
    .agg(
        sample_reviews=("recommendationid", "count"),
        positive_reviews=("steam_label_text", lambda x: (x == "positive").sum()),
        negative_reviews=("steam_label_text", lambda x: (x == "negative").sum()),
        high_urgency_reviews=("urgency", lambda x: (x == "high").sum()),
        avg_sentiment_score=("sentiment_score", "mean")
    )
    .reset_index()
    .sort_values("sample_reviews", ascending=False)
)

print("[게임별 샘플 수 요약]")
display(game_sample_summary)

print("[게임별 샘플 수 통계]")
display(game_sample_summary["sample_reviews"].describe())

[게임별 샘플 수 요약]


,appid,game_name,sample_reviews,positive_reviews,negative_reviews,high_urgency_reviews,avg_sentiment_score
4,1466060,Tainted Grail: The Fall of Avalon,100,50,50,44,3.100000
24,3027930,Karate Survivor,100,75,25,24,3.930000
16,2514460,GUARDS!,98,61,37,39,3.204082
6,1585180,Drova - Forsaken Kin,96,49,47,22,3.239583
11,2101890,Zoonomaly,87,54,33,29,3.137931
14,2488510,FatalZone,56,18,38,22,2.500000
3,1409200,HYPERVIOLENT,41,30,11,12,3.585366
0,1054510,Survivalist: Invisible Strain,38,29,9,11,3.763158
17,2617090,Fowl Damage,36,36,0,0,4.972222
23,3001200,Tails Noir: Rebel Rush,34,34,0,1,4.852941


[게임별 샘플 수 통계]


count     28.000000
mean      27.500000
std       35.806579
min        1.000000
25%        2.750000
50%        7.000000
75%       38.750000
max      100.000000
Name: sample_reviews, dtype: float64

# 9. 최종 품질 요약

앞에서 확인한 핵심 지표를 한 표로 정리한다.

In [10]:
# ============================================================
# 9. 최종 품질 요약
# ============================================================

quality_summary = pd.DataFrame({
    "항목": [
        "분석 결과 행 수",
        "분석 게임 수",
        "분석 성공 수",
        "중복 리뷰 ID 수",
        "Steam 긍정 리뷰 수",
        "Steam 부정 리뷰 수",
        "LLM exact_match 수",
        "LLM partial_match 수",
        "LLM unclear 수",
        "이슈 태그 없는 리뷰 수"
    ],
    "값": [
        len(df_result),
        df_result["appid"].nunique(),
        (df_result["analysis_status"] == "success").sum(),
        df_result["recommendationid"].duplicated().sum(),
        (df_result["steam_label_text"] == "positive").sum(),
        (df_result["steam_label_text"] == "negative").sum(),
        (df_result["sentiment_relation"] == "exact_match").sum(),
        (df_result["sentiment_relation"] == "partial_match").sum(),
        (df_result["sentiment_relation"] == "unclear").sum(),
        len(reviews_without_tags)
    ]
})

display(quality_summary)

,항목,값
0,분석 결과 행 수,770
1,분석 게임 수,28
2,분석 성공 수,770
3,중복 리뷰 ID 수,0
4,Steam 긍정 리뷰 수,511
5,Steam 부정 리뷰 수,259
6,LLM exact_match 수,704
7,LLM partial_match 수,58
8,LLM unclear 수,6
9,이슈 태그 없는 리뷰 수,0
